# Phase 2 — Components from scratch

We now build the model **one piece at a time**, each in its own file under
`transformer/`, and *feel* each one before assembling the whole thing in Phase 3:

- `positional_encoding.py` — sinusoidal PE (visualize the striped heatmap)
- `multi_head_attention.py` — scaled dot-product attention + multi-head wrapper
- `feed_forward.py` — the per-position non-linearity
- `encoder_layer.py` / `decoder_layer.py` — residual + LayerNorm (post-LN, paper)

These were ported from the sibling `attention-implementation` repo. Each file has a
`__main__` shape test (`uv run python -m transformer.<name>`). This notebook goes
further: the three feel-checks the plan calls for — the **PE heatmap**, **shape
preservation** through an encoder layer, and the **causal-mask perturbation test**
(the one that catches subtle masking bugs).

In [ ]:
# Bootstrap: put the repo root on sys.path so `import transformer` works from notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformer.config import ModelConfig
from transformer.positional_encoding import SinusoidalPositionalEncoding
from transformer.multi_head_attention import MultiHeadAttention
from transformer.feed_forward import PositionwiseFFN
from transformer.encoder_layer import EncoderLayer
from transformer.decoder_layer import DecoderLayer
from transformer.masks import build_tgt_mask, build_src_mask

torch.manual_seed(0)

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")
device = get_device()
print("device:", device)

## Scaled dot-product attention — the math

Given queries **Q** ∈ ℝ^(T×d_k), keys **K** ∈ ℝ^(S×d_k), values **V** ∈ ℝ^(S×d_v):

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

**Why √d_k?**  Each element of *QKᵀ* is a dot product of two d_k-dimensional vectors
whose entries are roughly unit-variance. The variance of that dot product grows as d_k
(sum of d_k independent unit-variance terms), so its standard deviation is √d_k.
Without the scaling, large d_k drives dot products into the flat, saturated tails of
softmax — where gradients vanish and every head collapses to a near-uniform distribution.
Dividing by √d_k keeps the pre-softmax scores at ~unit variance regardless of d_k.

**Multi-head attention** runs *h* heads in parallel, each on a d_k = d_model/h projection:

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\, W^O$$

where head_i = Attention(Q W_i^Q,\; K W_i^K,\; V W_i^V).

The paper's base config uses d_model=512, h=8, so d_k=64 per head.
Each head specialises in a different subspace — positional proximity in one head,
syntactic agreement in another, coreference in a third, and so on.


## Feel-check 1: the positional-encoding heatmap

Plot PE for `d_model=256, max_len=100`. Each row is a position, each column a
dimension. You should see the famous **striped / interleaved sine-cosine pattern**:
low dimensions oscillate slowly (long wavelength), high dimensions fast. This is the
"clock with many hands" that lets attention recover both absolute and relative position.

In [ ]:
pe = SinusoidalPositionalEncoding(d_model=256, max_len=100)
mat = pe.pe.squeeze(0).numpy()  # (100, 256)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(mat, aspect="auto", cmap="RdBu")
ax.set_xlabel("embedding dimension"); ax.set_ylabel("position")
ax.set_title("Sinusoidal positional encoding (d_model=256)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

# A single dimension is a clean sinusoid; its frequency depends on the dim index.
fig, ax = plt.subplots(figsize=(8, 3))
for d in (0, 4, 20, 100):
    ax.plot(mat[:, d], label=f"dim {d}")
ax.set_xlabel("position"); ax.set_ylabel("value"); ax.legend(ncol=4)
ax.set_title("Low dims wave slowly, high dims wave fast")
plt.tight_layout(); plt.show()

## Feel-check 2: one encoder layer preserves shape

The whole Transformer is a stack of shape-preserving blocks: `(B, T, d_model)` in,
`(B, T, d_model)` out. Confirm it for a single encoder layer on a real-ish batch.

In [ ]:
cfg = ModelConfig.smoke()          # small config: d_model=128, h=4, 2+2 layers
cfg.dropout = 0.0                  # deterministic for inspection
print("cfg:", cfg)

enc = EncoderLayer(cfg).to(device).eval()
x = torch.randn(3, 9, cfg.d_model, device=device)
out = enc(x, src_mask=None)
print("in :", tuple(x.shape))
print("out:", tuple(out.shape))
assert out.shape == x.shape
print("shape preserved ✓")

## Multi-head attention + a look at the attention map

`MultiHeadAttention` has a `cache_attn` hook (off by default, zero cost during
training). Turn it on to grab the `(B, h, Tq, Tk)` attention weights and visualize
them. With a causal mask, the map must be **lower-triangular** (each query only puts
weight on itself and earlier keys) and **each row sums to 1** (softmax).

In [ ]:
mha = MultiHeadAttention(cfg).to(device).eval()
mha.cache_attn = True

T = 8
x = torch.randn(1, T, cfg.d_model, device=device)
tgt_pad = torch.zeros(1, T, dtype=torch.bool, device=device)
mask = build_tgt_mask(tgt_pad, use_causal=True)   # (1,1,T,T) additive
_ = mha(x, x, x, mask=mask)
attn = mha.last_attn[0]   # (h, T, T)

print("attention shape (h, Tq, Tk):", tuple(attn.shape))
print("row sums (should be ~1):", attn[0].sum(-1).cpu().numpy().round(3))

fig, axes = plt.subplots(1, min(4, cfg.n_heads), figsize=(12, 3))
for hd, ax in enumerate(axes):
    ax.imshow(attn[hd].cpu(), cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"head {hd}"); ax.set_xlabel("key"); ax.set_ylabel("query")
fig.suptitle("Causal self-attention is lower-triangular (upper triangle = 0)")
plt.tight_layout(); plt.show()
assert torch.allclose(attn.sum(-1), torch.ones_like(attn.sum(-1)), atol=1e-5)
# Upper triangle (future) must be exactly zero after the causal mask.
upper = torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)
assert attn[:, upper].abs().max() < 1e-6, "future positions leaked!"
print("no future leakage ✓")

### Reading the attention maps

With random weights the maps look noisy — no interpretable structure yet. A few things
to notice even here:

- **Lower-triangular only** — no query has weight above the diagonal. The causal mask
  is working.
- **Rows sum to 1** — softmax normalisation is intact (printed above).
- **Heads differ slightly** — even random W_Q, W_K, W_V projections produce different
  score distributions across heads. After training each head will specialise: some
  attend locally (adjacent tokens), others globally (sentence-level dependencies).
- **No attention to pad** — if there were pad tokens in this toy sequence, their
  columns would be zeroed out by the mask before softmax.

The attention pattern is the primary diagnostic tool during and after training.
Plotting a few heads on a real translation (`05_generation.ipynb`) is how you verify
the model actually learned alignment.


## Feel-check 3: the causal-mask perturbation test (the important one)

This is the test that catches subtle masking bugs. Run a decoder input through a
decoder layer (with causal mask). Then **perturb a single token at position `i`** and
run again. Because of the causal mask, position `t` can only attend to positions
`<= t`, so:

- outputs at positions `< i` must be **unchanged** (they never see token `i`);
- outputs at positions `>= i` **may change**.

If positions before `i` change, the mask is leaking the future and the model is broken.

In [ ]:
torch.manual_seed(1)
dec = DecoderLayer(cfg).to(device).eval()   # eval + dropout=0 -> deterministic

T, S = 10, 7
x = torch.randn(1, T, cfg.d_model, device=device)
memory = torch.randn(1, S, cfg.d_model, device=device)
tgt_pad = torch.zeros(1, T, dtype=torch.bool, device=device)
tgt_mask = build_tgt_mask(tgt_pad, use_causal=True)

out1 = dec(x, memory, tgt_mask=tgt_mask, memory_mask=None)

i = 4  # perturb this token
x2 = x.clone()
x2[:, i, :] += 5.0  # large perturbation so any leak is obvious
out2 = dec(x2, memory, tgt_mask=tgt_mask, memory_mask=None)

per_pos_change = (out2 - out1).abs().amax(dim=-1)[0]  # (T,)
print(f"perturbed position i={i}")
for t, c in enumerate(per_pos_change.tolist()):
    flag = "  <- before i (must be ~0)" if t < i else ""
    print(f"  pos {t:2d}: max |delta| = {c:.3e}{flag}")

assert per_pos_change[:i].max() < 1e-5, "LEAK: a position before i changed!"
assert per_pos_change[i:].max() > 1e-3, "expected positions >= i to change"
print("\ncausal mask is correct: only positions >= i changed ✓")

## Position-wise FFN — the only per-position non-linearity

Attention mixes information *across* positions but is linear in V. The FFN
(`Linear -> ReLU -> Linear`) is the only place a position can apply a non-linear
transform to its own representation — and it's the bulk of the parameters.

In [ ]:
ffn = PositionwiseFFN(cfg).to(device).eval()
x = torch.randn(3, 9, cfg.d_model, device=device)
out = ffn(x)
assert out.shape == x.shape
n_params = sum(p.numel() for p in ffn.parameters())
print(f"FFN out shape: {tuple(out.shape)}")
print(f"FFN params: {n_params:,}  (d_model={cfg.d_model} -> d_ff={cfg.d_ff} -> d_model)")

## Takeaways

- PE is a fixed bank of sinusoids; the heatmap stripes are real.
- Every block preserves `(B, T, d_model)`.
- The causal mask provably stops information flowing backward in time — the
  perturbation test is your regression check whenever you touch masking.

**Next:** `03_full_model.ipynb` — stack these into `Encoder`, `Decoder`, and the full
`Transformer` (embeddings + PE + weight tying + output projection), count parameters,
and verify the **untrained loss ≈ log(vocab_size)** sanity check before any training.